# Event-Driven Federated Learning — Five Approach Comparison

Run notebooks 01–05 in order first, in this folder, so their CSV metric exports are available. The order is synchronous wait-for-all, async immediate no buffer, async weighted staleness, async unweighted staleness K=5, then async weighted staleness K=5. This notebook loads the five runs and plots accuracy/loss against client updates and virtual time. All curves use the recorded evaluations; asynchronous curves are evaluated after each server update.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

FILES = {
    'Synchronous wait-for-all': 'round_metrics.csv',
    'Async no buffer': 'async_immediate_no_buffer_metrics.csv',
    'Async staleness weighted': 'async_staleness_weighted_no_buffer_metrics.csv',
    'Async staleness unweighted K=5': 'async_staleness_unweighted_k5_metrics.csv',
    'Async staleness weighted K=5': 'async_staleness_weighted_k5_metrics.csv',
}


In [ ]:
series = {}
missing = []
for name, filename in FILES.items():
    path = Path(filename)
    if not path.exists():
        missing.append(filename)
        continue
    df = pd.read_csv(path)
    if 'round' in df.columns:  # synchronous schema
        df = df.rename(columns={'round':'step','end_time':'time'})
        df['updates'] = df['step'] * 10
    else:  # asynchronous schema
        df = df.rename(columns={'aggregation':'step'})
        df['time'] = df['end_time']
        df['updates'] = df['updates_received']
    required = {'step', 'time', 'updates', 'accuracy', 'loss'}
    absent = required - set(df.columns)
    if absent:
        raise ValueError(f'{filename} is missing required columns: {sorted(absent)}')
    if df.empty or df[['time', 'updates', 'accuracy', 'loss']].isna().any().any():
        raise ValueError(f'{filename} is empty or contains missing plotted metrics.')
    series[name] = df
if missing:
    raise FileNotFoundError('Cannot plot all five approaches. Missing exports (run notebooks 01–05): ' + ', '.join(missing))
summary=[]
for name, df in series.items():
    final=df.iloc[-1]
    summary.append({'approach':name,'evaluations':len(df),'updates':int(final['updates']),
                    'virtual_time_s':float(final['time']),'final_accuracy_pct':100*float(final['accuracy']),
                    'final_loss':float(final['loss'])})
summary_df=pd.DataFrame(summary)
display(summary_df.round(4))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14,5))
for name, df in series.items():
    axes[0].plot(df['updates'], df['accuracy']*100, marker='o', markersize=3, label=name)
    axes[1].plot(df['time'], df['accuracy']*100, marker='o', markersize=3, label=name)
axes[0].set(xlabel='Cumulative client updates', ylabel='Test accuracy (%)', title='Accuracy vs client updates')
axes[1].set(xlabel='Elapsed virtual time (s)', ylabel='Test accuracy (%)', title='Accuracy vs simulated time')
for ax in axes:
    ax.grid(alpha=.3)
    ax.legend(loc='best', fontsize=8)
plt.tight_layout(); plt.show()

plt.figure(figsize=(9,5))
for name, df in series.items(): plt.plot(df['time'], df['loss'], marker='o', markersize=3, label=name)
plt.xlabel('Elapsed virtual time (s)'); plt.ylabel('Test loss'); plt.title('Test loss vs simulated time')
plt.grid(alpha=.3); plt.legend(fontsize=8); plt.tight_layout(); plt.show()

plt.figure(figsize=(9,5))
for name, df in series.items():
    if 'cumulative_throughput' in df.columns:
        plt.plot(df['time'], df['cumulative_throughput'], marker='o', markersize=3, label=name)
    elif 'updates' in df.columns:
        throughput=df['updates']/df['time'].where(df['time']>0)
        plt.plot(df['time'], throughput, marker='o', markersize=3, label=name)
plt.xlabel('Elapsed virtual time (s)'); plt.ylabel('Cumulative updates / second'); plt.title('Throughput vs simulated time')
plt.grid(alpha=.3); plt.legend(fontsize=8); plt.tight_layout(); plt.show()
summary_df.to_csv('five_approach_comparison_summary.csv',index=False)
print('Saved five_approach_comparison_summary.csv')
